In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import umap
import pickle

In [8]:
df = pd.read_csv('players_data_light-2024_2025.csv')
df = df[df['Min'] >= 450].copy() # filters players who have more than 450 minutes played
outfield = df[df['Pos'] != 'GK'].copy().reset_index(drop=True) # seperated goalkeepers due to different attributes

In [9]:
# defining feature columns and removing redundancies
GK_COLS = [
    'GA','GA90','SoTA','Saves','Save%','W','D','L','CS','CS%',
    'PKatt_stats_keeper','PKA','PKsv','PKm','PSxG','PSxG/SoT',
    'PSxG+/-','/90','Cmp_stats_keeper_adv','Att_stats_keeper_adv',
    'Cmp%_stats_keeper_adv','Att (GK)','Thr','Launch%','AvgLen',
    'Opp','Stp','Stp%','#OPA','#OPA/90','AvgDist'
]
META_COLS = [
    'Rk','Player','Nation','Pos','Squad','League','Age','Born',
    'Matches Played','Starts','Min','90s'
]
REDUNDANT_COLS = [
    'G+A','G+A-PK','xG+xAG','npxG+xAG','Tkl+Int',
    'PK_stats_shooting','PKatt_stats_shooting',              
    'xG_stats_shooting','npxG_stats_shooting',
    'Ast_stats_passing','xAG_stats_passing','PrgP_stats_passing',
    'FK_stats_passing_types','Cmp_stats_passing_types','Att_stats_defense',
    'Fld_stats_misc','Off_stats_misc','Crs_stats_misc','Int_stats_misc',
    'TklW_stats_misc','CrdY_stats_misc','CrdR_stats_misc','Lost_stats_misc',
    'PrgC_stats_possession','PrgR_stats_possession',
    'TotDist_stats_possession','PrgDist_stats_possession',
    'Live_stats_possession','Att_stats_possession',
    'Sh_stats_defense','Sh_stats_gca','PassLive','PassDead',
]

EXCLUDE = set(GK_COLS + META_COLS + REDUNDANT_COLS)
numeric_cols = outfield.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in EXCLUDE]

In [10]:
# Normalization
# stats that are already scaled to per 90 minutes
rates = {
    'SoT%','Sh/90','SoT/90','G/Sh','G/SoT','npxG/Sh',
    'G-xG','np:G-xG','Cmp%','Tkl%','Succ%','Tkld%','Won%','A-xAG'
}

X = outfield[feature_cols].copy()
nineties = outfield['90s'].values
# scaled to stats per 90 minutes
for col in feature_cols:
    if col not in rates:
        X[col] = X[col] / nineties
        
#imputed empty stats with 0
X_imp    = SimpleImputer(strategy='constant', fill_value = 0).fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_imp)
X_pca    = PCA(n_components=0.90, random_state=42).fit_transform(X_scaled) #PCA to retain 90% of the variance

In [11]:
# K means clustering
K = 8 
km = KMeans(n_clusters=K, random_state=42, n_init=20)
outfield['cluster'] = km.fit_predict(X_pca)

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=20, min_dist=0.1)
embedding = reducer.fit_transform(X_pca)
outfield['umap_x'] = embedding[:, 0]
outfield['umap_y'] = embedding[:, 1]

C:\Users\admin\Downloads\det project\detmp\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [22]:
##Test for the main file
# Similar players
def find_similar(player_name, n=5):
    matches = outfield[outfield['Player'].str.contains(player_name, case=False)]
    if matches.empty:
        print(f"Player '{player_name}' not found.")
        return
    idx = matches.index[0]
    target = outfield.loc[idx]
    distances = np.sqrt(
        (outfield['umap_x'] - target['umap_x'])**2 +
        (outfield['umap_y'] - target['umap_y'])**2
    )
    similar = outfield.loc[distances.nsmallest(n + 1).index[1:]]
    print(f"\nMost similar to {target['Player']} ({target['Squad']}):\n")
    print(similar[['Player','League','Squad','Pos']].to_string(index=False))

find_similar('Saka') #test


Most similar to Bukayo Saka (Arsenal):

       Player         League       Squad   Pos
      Rodrygo        La Liga Real Madrid FW,MF
Florian Wirtz     Bundesliga  Leverkusen MF,FW
Matheus Cunha Premier League      Wolves MF,FW
  Désiré Doué        Ligue 1   Paris S-G FW,MF
Son Heung-min Premier League   Tottenham    FW


In [23]:
with open('model.pkl', 'wb') as f:
    pickle.dump({
        'outfield': outfield,  # has cluster, umap_x, umap_y already attached
        'km': km,
        'feature_cols': feature_cols
    }, f)

NameError: name 'pickle' is not defined